# 03 - EDA and Feature Selection

## Purpose
Inspect the engineered feature and target files before any modeling work.

## Inputs
- Per-sample `gee_features.csv`
- Per-sample `gee_targets.csv`
- `data/processed/sample_index.csv`

## Outputs
- In-notebook summaries for class balance, missingness, target health, and feature correlations

## Notes
This notebook is for analysis only. It does not train models and does not write new dataset files.

## 1. Configure EDA Paths
Defines the project root and sample index path used to locate all per-sample feature and target files.

In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAMPLE_INDEX_PATH = PROJECT_ROOT / "data" / "processed" / "sample_index.csv"

## 2. Load Per-Sample Features and Targets
Reloads the feature helper module and stacks per-sample feature/target files in memory for analysis only.

In [ ]:
import src.features.gee_features as gee_feature_module
importlib.reload(gee_feature_module)

features = gee_feature_module.load_per_sample_features(SAMPLE_INDEX_PATH)
targets = gee_feature_module.load_per_sample_targets(SAMPLE_INDEX_PATH)
print("features", features.shape)
print("targets", targets.shape)

## 3. Check Class Balance and Missingness
Reports sample-level class counts and the highest missing-value rates in both features and targets.

In [ ]:
class_balance = features[["sample_id", "label"]].drop_duplicates()["label"].value_counts()
missing_rate = features.isna().mean().sort_values(ascending=False)
target_missing_rate = targets.isna().mean().sort_values(ascending=False)

print("Class balance:")
print(class_balance)
print("Top feature missing rates:")
print(missing_rate.head(15))
print("Target missing rates:")
print(target_missing_rate.head(15))

## 4. Compute Numeric Feature Correlations
Builds a numeric correlation matrix to support later feature-selection decisions.

In [ ]:
numeric_features = features.select_dtypes(include="number")
correlation = numeric_features.corr(numeric_only=True)
correlation.head()